### Загрузка данных

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [17]:
df = pd.read_csv('../../models/dataset.csv', index_col=0)
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6


In [3]:
df_features = pd.read_csv('../../data/features.csv')
df_features.head()

,Регион,Год,Заболеваемость_инфекции_на_1000,Коэффициент_смертности_населения,Численность_населения,Оборотная_вода_млн_м3
0,Белгородская область,2004,41.3,16.2,1511.7,1610.0
1,Брянская область,2004,33.8,19.1,1344.1,63.0
2,Владимирская область,2004,46.5,20.1,1497.6,339.0
3,Воронежская область,2004,26.1,18.5,2364.9,2419.0
4,Ивановская область,2004,31.6,21.6,1116.7,223.0


### Merge df_main + df_features

#### Проверка правильности регионов

In [7]:
def compare_unique(df1, df2):
    shared = set(df1.columns) & set(df2.columns)
    differences = {
        col: {
            'only_in_df1': sorted(set(df1[col].unique()) - set(df2[col].unique())),
            'only_in_df2': sorted(set(df2[col].unique()) - set(df1[col].unique()))
        }
        for col in shared
        if set(df1[col].unique()) != set(df2[col].unique())
    }
    identical = [col for col in shared if set(df1[col].unique()) == set(df2[col].unique())]
    if 'Регион' in identical:
        print("Различий уникальных значений по столбцу 'Регион' нет.")
    elif 'Регион' in differences:
        print("Есть различия по уникальным значениям в столбце 'Регион'.")
    return differences, identical


In [8]:
compare_unique(df_main, df_features)

Различий уникальных значений по столбцу 'Регион' нет.


({}, ['Год', 'Регион'])

#### Соединяем таблицы по двум ключам (inner join)

In [10]:
df_main.shape

(1992, 5)

In [11]:
df_features.shape

(1992, 6)

In [12]:
df = pd.merge(
    df_main, df_features,
    on=['Регион', 'Год'],
    how='inner',  
    suffixes=('_df1', '_df2')
)


In [13]:
df.rename(columns={
    'Объем_сточных_вод_млн_м3': 'V_сточ_вод_млн_м3',
    'Инвестиции_в_ООС': 'Инв_в_ООС',
    'ВРП': 'ВРП',
    'Заболеваемость_инфекции_на_1000': 'Забол_инф_на_1000',
    'Коэффициент_смертности_населения': 'Коэф_смерт_насел',
    'Численность_населения': 'Числ_насел',
    'Оборотная_вода_млн_м3': 'Оборот_вод_млн_м3'
}, inplace=True)

In [19]:
df.columns

Index(['Регион', 'Год', 'Объем_сточных_вод_млн_м3', 'Инвестиции_в_ООС_тыс_руб',
       'ВРП_млн_руб', 'ВРП_на_душу_руб', 'Индексы_производства_продукции_СХ_%',
       'Доля_городского_населения_%', 'Использование_свеж_воды_млн_м3',
       'Индекс_промыш_производства_%'],
      dtype='object')

In [21]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6


In [23]:
df.shape

(1992, 10)

In [25]:
df.isna().sum()

Регион                                  0
Год                                     0
Объем_сточных_вод_млн_м3                5
Инвестиции_в_ООС_тыс_руб                5
ВРП_млн_руб                             5
ВРП_на_душу_руб                        36
Индексы_производства_продукции_СХ_%    42
Доля_городского_населения_%             0
Использование_свеж_воды_млн_м3          0
Индекс_промыш_производства_%           21
dtype: int64

In [18]:
# В стобце "Забол_инф_на_1000" много пропусков, удалим его
df = df.drop('Забол_инф_на_1000', axis=1)
df.head()

,Регион,Год,V_сточ_вод_млн_м3,Инв_в_ООС,ВРП,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,176660.0,46736.8,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,111590.0,61854.4,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,67466.0,73107.4,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,50114.0,88733.3,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,38270.0,114840.5,15.9,2539.4,969.0


In [19]:
df.shape

(1992, 8)

#### Удаляем регион с пропусками

In [27]:
df = df[df['Регион'] != 'Чеченская Республика']

#### Меняем формат числа у ВРП в 2023 году

In [30]:
df[df['Год'] == 2023] # сейчас данные находятся в тыс., а нужно в миллионах, как у остальных

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
23,Алтайский край,2023,13.0,2247651.0,1.024355e+06,482474.4,93.1,58.5,425.0,107.1
47,Амурская область,2023,60.0,2883951.0,7.938519e+05,1054056.2,94.2,68.5,115.0,97.2
71,Архангельская область,2023,247.0,2286387.0,7.615896e+05,793259.7,101.5,78.1,500.0,98.8
95,Астраханская область,2023,85.0,191976.0,7.777183e+05,819951.6,103.0,63.9,561.0,100.6
119,Белгородская область,2023,58.0,17411221.0,1.341409e+06,889768.6,105.0,65.3,224.0,104.7
...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,1.867094e+05,3895053.5,91.5,69.4,100.0,110.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5.379402e+06,10462220.5,80.7,85.2,189.0,97.1
1943,Ярославская область,2023,143.0,9917079.0,8.497699e+05,713444.2,105.1,80.8,180.0,107.0
1967,г. Москва,2023,833.0,22791947.0,3.233900e+07,2463550.4,70.8,100.0,1331.0,119.0


In [53]:
df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] = df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] / 1

In [55]:
df['ВРП_млн_руб'] = df['ВРП_млн_руб'].round(1)

In [57]:
df[df['Год'] == 2023]

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
23,Алтайский край,2023,13.0,2247651.0,1024355.4,482474.4,93.1,58.5,425.0,107.1
47,Амурская область,2023,60.0,2883951.0,793851.9,1054056.2,94.2,68.5,115.0,97.2
71,Архангельская область,2023,247.0,2286387.0,761589.6,793259.7,101.5,78.1,500.0,98.8
95,Астраханская область,2023,85.0,191976.0,777718.3,819951.6,103.0,63.9,561.0,100.6
119,Белгородская область,2023,58.0,17411221.0,1341408.9,889768.6,105.0,65.3,224.0,104.7
...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,186709.4,3895053.5,91.5,69.4,100.0,110.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5379401.8,10462220.5,80.7,85.2,189.0,97.1
1943,Ярославская область,2023,143.0,9917079.0,849769.9,713444.2,105.1,80.8,180.0,107.0
1967,г. Москва,2023,833.0,22791947.0,32339001.6,2463550.4,70.8,100.0,1331.0,119.0


#### Нормализуем Инвестии_в_ООС по ВРП

In [65]:
df['Инвестиции_в_ООС_тыс_руб'] = df['Инвестиции_в_ООС_тыс_руб'] / df['ВРП_млн_руб']

In [67]:
df = df.drop(['Инвестиции_в_ООС_тыс_руб', 'ВРП_млн_руб'], axis=1)

In [69]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6


### Сохранение датасета

In [72]:
df.describe()

,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
count,1968.000000,1968.000000,1.937000e+03,1931.000000,1968.000000,1968.000000,1954.000000
mean,2011.500000,189.902536,4.223254e+05,102.279751,70.151067,691.476479,105.074539
std,6.923946,266.510499,8.568329e+05,10.808069,12.567786,941.456301,10.109419
min,2000.000000,0.000000,6.667900e+03,41.400000,26.000000,4.720000,43.200000
25%,2005.750000,39.715000,9.443650e+04,97.100000,63.675000,133.490000,100.700000
50%,2011.500000,88.805000,2.282500e+05,101.600000,70.650000,294.000000,104.400000
75%,2017.250000,216.805000,4.392505e+05,106.400000,77.800000,799.000000,109.100000
max,2023.000000,2661.000000,1.199539e+07,185.000000,100.000000,6849.000000,273.700000


In [74]:
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6


In [78]:
df.to_excel('../../data/dataset_11_11.xlsx')
df.to_csv('../../data/dataset_11_11.csv')